# mBART Pipeline:
Input text -> Encoder -> Decoder -> Generated output

# Explanation:
mBART is a multilingual sequence-to-sequence model based on the BART chitecture.
It is designed to work across multiple languages.
Unlike DistilBART, mBART is not compressed; it is larger and usually slower,
but it can support multilingual input.

# Work:
Input text  
   ↓  
mBART (Transformers + PyTorch)  
   ↓  
Summary / key content  
   ↓  
spaCy  
   ↓  
Entities: contributors, organizations, dates, etc.

In [145]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
import spacy
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

In [160]:
# ============================================================
# Configuration
# ============================================================
INPUT_JSON = "../Data/silver/doc_01_silver_nlp.json"
OUTPUT_DIR = "../Data/gold/mBART"
OUTPUT_FILE = "mBART_output.json"

TEXT_COLUMN = "cleaned_text"
DOC_ID_COLUMN = "document_id"
SENTENCES_COLUMN = "sentences"

MBART_MODEL = "facebook/mbart-large-50"
SRC_LANG = "en_XX"

ZERO_SHOT_MODEL = "facebook/bart-large-mnli"

MAX_INPUT_LEN = 1024
MAX_OUTPUT_LEN = 170
MIN_OUTPUT_LEN = 110

MAX_SIMILARITY = 0.68
MAX_SELECTED_SENTENCES = 12
MAX_KEY_ENTITIES = 15

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]
RESEARCH_GROUP_CONFIDENCE_THRESHOLD = 0.20

In [147]:
# ============================================================
# Load models
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    MBART_MODEL,
    src_lang=SRC_LANG,
    use_fast=False
)
model = AutoModelForSeq2SeqLM.from_pretrained(MBART_MODEL).to(device)
model.eval()

nlp = spacy.load("en_core_web_sm")

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model=ZERO_SHOT_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

Loading weights: 100%|██████████| 519/519 [00:00<00:00, 35809.83it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 15492.12it/s]


In [ ]:
# ============================================================
# Generic helpers
# ============================================================
def safe_string(value: Any) -> str:
    return "" if value is None else str(value)


def normalize_whitespace(text: Any) -> str:
    return re.sub(r"\s+", " ", safe_string(text)).strip()


def simple_tokenize(text: Any) -> List[str]:
    return re.findall(r"\b\w+\b", safe_string(text).lower())


def clean_text(text: str) -> str:
    text = "Summarize the following document in a concise abstract form: " + text
    text = text.replace("\x00", " ").replace("\ufeff", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    cleaned_lines = []
    for line in text.splitlines():
        line_strip = line.strip()

        if re.fullmatch(r"\d{1,4}", line_strip):
            continue
        if re.fullmatch(r"page\s+\d+(\s+of\s+\d+)?", line_strip, flags=re.IGNORECASE):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def split_sentences(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    doc = nlp(text)
    sentences = [normalize_whitespace(sent.text) for sent in doc.sents]
    return [s for s in sentences if s]

In [149]:
# ============================================================
# Noise / heading detection
# ============================================================
SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents"
}

INSTITUTION_PATTERNS = [
    r"\buniversity\b",
    r"\bcollege\b",
    r"\bfaculty\b",
    r"\bschool\b",
    r"\binstitute\b",
    r"\bdepartment\b",
    r"\bapplied sciences\b",
    r"\bhz university\b"
]

AUTHOR_PREFIX_PATTERNS = [
    r"^(author|authors|written by|student|students|name)\s*[:\-]?\s+",
    r"^by\s+[A-Z]"
]

TITLE_PREFIX_PATTERNS = [
    r"^(title|report title|project title)\s*[:\-]\s*"
]


def remove_section_prefix(sentence: str) -> str:
    s = normalize_whitespace(sentence)
    s = re.sub(
        r"^(abstract|introduction|background|discussion|conclusion|conclusions|results|result|method|methods|methodology|summary)\b[:\-\s]*",
        "",
        s,
        flags=re.IGNORECASE
    )
    return normalize_whitespace(s)


def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if "table of contents" in s.lower():
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.search(r"\.{4,}", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    if re.search(r"\b\d+\.\d+\b", s) and len(s) < 120:
        return True

    return False


def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True

    return False


def is_heading_like(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if len(s.split()) <= 5 and s.istitle():
        return True
    if re.fullmatch(r"[A-Z][A-Za-z\s/&\-]{1,60}", s) and len(s.split()) <= 8:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True

    return False


def clean_sentence_for_selection(sentence: str) -> str:
    s = remove_section_prefix(sentence)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_good_sentence(sentence: str) -> bool:
    s = clean_sentence_for_selection(sentence)

    if len(s) < 45:
        return False
    if len(s.split()) < 8:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if is_heading_like(s):
        return False
    if re.match(r"^\d+(\.\d+)*\s*$", s):
        return False
    if "................................................................" in s:
        return False
    if "•" in s and len(s.split()) < 15:
        return False

    return True


In [150]:
# ============================================================
# Noise / heading detection
# ============================================================
SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents"
}

INSTITUTION_PATTERNS = [
    r"\buniversity\b",
    r"\bcollege\b",
    r"\bfaculty\b",
    r"\bschool\b",
    r"\binstitute\b",
    r"\bdepartment\b",
    r"\bapplied sciences\b",
    r"\bhz university\b"
]

AUTHOR_PREFIX_PATTERNS = [
    r"^(author|authors|written by|student|students|name)\s*[:\-]?\s+",
    r"^by\s+[A-Z]"
]

TITLE_PREFIX_PATTERNS = [
    r"^(title|report title|project title)\s*[:\-]\s*"
]


def remove_section_prefix(sentence: str) -> str:
    s = normalize_whitespace(sentence)
    s = re.sub(
        r"^(abstract|introduction|background|discussion|conclusion|conclusions|results|result|method|methods|methodology|summary)\b[:\-\s]*",
        "",
        s,
        flags=re.IGNORECASE
    )
    return normalize_whitespace(s)


def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if "table of contents" in s.lower():
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.search(r"\.{4,}", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    if re.search(r"\b\d+\.\d+\b", s) and len(s) < 120:
        return True

    return False


def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True

    return False


def is_heading_like(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if len(s.split()) <= 5 and s.istitle():
        return True
    if re.fullmatch(r"[A-Z][A-Za-z\s/&\-]{1,60}", s) and len(s.split()) <= 8:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True

    return False


def clean_sentence_for_selection(sentence: str) -> str:
    s = remove_section_prefix(sentence)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_good_sentence(sentence: str) -> bool:
    s = clean_sentence_for_selection(sentence)

    if len(s) < 45:
        return False
    if len(s.split()) < 8:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if is_heading_like(s):
        return False
    if re.match(r"^\d+(\.\d+)*\s*$", s):
        return False
    if "................................................................" in s:
        return False
    if "•" in s and len(s.split()) < 15:
        return False

    return True


In [151]:
# ============================================================
# Ranking and sentence selection
# ============================================================
def jaccard_similarity(a: str, b: str) -> float:
    set_a = set(simple_tokenize(a))
    set_b = set(simple_tokenize(b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def sentence_position_score(index: int, total: int) -> float:
    if total <= 0:
        return 0.0

    relative = index / max(total, 1)

    if relative < 0.10:
        return 0.9
    if relative < 0.25:
        return 0.5
    if relative > 0.90:
        return 0.4

    return 0.0


def sentence_information_score(sentence: str) -> float:
    s = sentence.lower()
    word_count = len(sentence.split())
    score = 0.0

    if 12 <= word_count <= 35:
        score += 1.0
    elif 36 <= word_count <= 50:
        score += 0.6
    elif word_count > 55:
        score -= 0.5

    discourse_terms = [
        "this study", "this report", "this paper", "the aim", "the purpose",
        "the objective", "the analysis", "the findings", "the results",
        "shows that", "demonstrates that", "indicates that", "suggests that",
        "because", "therefore", "however", "overall", "in conclusion"
    ]
    for term in discourse_terms:
        if term in s:
            score += 0.5

    tokens = simple_tokenize(sentence)
    if tokens:
        unique_ratio = len(set(tokens)) / len(tokens)
        score += 0.8 * unique_ratio

    vague_phrases = [
        "this is needed", "it also makes", "it means that", "there are a lot of"
    ]
    for phrase in vague_phrases:
        if phrase in s:
            score -= 0.5

    return score


def classify_sentence_role(sentence: str) -> str:
    s = sentence.lower()

    if any(x in s for x in [
        "this study", "this report", "this paper", "the aim", "the purpose",
        "we examine", "we explore", "we investigate", "focus on", "focuses on"
    ]):
        return "objective"

    if any(x in s for x in [
        "method", "approach", "analysis", "using", "based on",
        "evaluate", "evaluates", "assessment", "assesses", "analyzes", "examines"
    ]):
        return "method"

    if any(x in s for x in [
        "results", "findings", "shows that", "demonstrates that",
        "indicates that", "reveals that", "suggests that", "is reduced"
    ]):
        return "findings"

    if any(x in s for x in [
        "recommend", "should", "could", "future work", "implementation",
        "strategy", "plan", "timeline", "metrics for success"
    ]):
        return "recommendation"

    if any(x in s for x in [
        "in conclusion", "overall", "to conclude", "in summary"
    ]):
        return "conclusion"

    return "background"


def rank_sentences(sentences: List[str]) -> List[Tuple[str, float]]:
    cleaned_candidates = []
    for i, s in enumerate(sentences):
        cleaned = clean_sentence_for_selection(s)
        if is_good_sentence(cleaned):
            cleaned_candidates.append((i, cleaned))

    total = len(cleaned_candidates)
    scored = []

    for i, sentence in cleaned_candidates:
        score = sentence_information_score(sentence) + sentence_position_score(i, total)
        scored.append((sentence, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


def select_balanced_sentences(
    sentences: List[str],
    max_sentences: int = MAX_SELECTED_SENTENCES,
    max_similarity: float = MAX_SIMILARITY,
    max_input_tokens: int = MAX_INPUT_LEN
) -> List[str]:
    ranked = rank_sentences(sentences)

    buckets = {
        "objective": [],
        "method": [],
        "findings": [],
        "recommendation": [],
        "conclusion": [],
        "background": []
    }

    for sent, _ in ranked:
        buckets[classify_sentence_role(sent)].append(sent)

    selected = []

    target_plan = [
        ("objective", 2),
        ("method", 2),
        ("findings", 2),
        ("recommendation", 2),
        ("conclusion", 1),
        ("background", 3),
    ]

    for role, limit in target_plan:
        count = 0
        for sent in buckets[role]:
            if any(jaccard_similarity(sent, prev) > max_similarity for prev in selected):
                continue

            candidate_text = " ".join(selected + [sent])
            token_count = len(tokenizer(candidate_text, truncation=False)["input_ids"])
            if token_count > max_input_tokens - 30:
                continue

            selected.append(sent)
            count += 1
            if count >= limit or len(selected) >= max_sentences:
                break

    if len(selected) < max_sentences:
        for sent, _ in ranked:
            if sent in selected:
                continue
            if any(jaccard_similarity(sent, prev) > max_similarity for prev in selected):
                continue

            candidate_text = " ".join(selected + [sent])
            token_count = len(tokenizer(candidate_text, truncation=False)["input_ids"])
            if token_count > max_input_tokens - 30:
                continue

            selected.append(sent)
            if len(selected) >= max_sentences:
                break

    ordered = []
    cleaned_source = [clean_sentence_for_selection(s) for s in sentences]
    selected_set = set(selected)

    for s in cleaned_source:
        if s in selected_set and s not in ordered:
            ordered.append(s)

    return ordered[:max_sentences]


def build_summary_input(selected_sentences: List[str]) -> str:
    cleaned = []

    for s in selected_sentences:
        s = clean_sentence_for_selection(s)
        if is_heading_like(s):
            continue
        cleaned.append(s)

    text = " ".join(cleaned)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [152]:
# ============================================================
# Date helpers
# ============================================================
def is_date_like_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False

    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\s*[-–]\s*\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}\s*[-–]\s*\d{4}-\d{2}-\d{2}", s):
        return True

    return False


def parse_two_digit_year(year_2d: int) -> int:
    if 0 <= year_2d <= 30:
        return 2000 + year_2d
    return 1900 + year_2d


def is_valid_date_parts(day: int, month: int, year: int) -> bool:
    current_year = time.localtime().tm_year

    if month < 1 or month > 12:
        return False
    if day < 1 or day > 31:
        return False
    if year < 1900 or year > current_year + 1:
        return False

    return True


def extract_dates(text: str) -> List[str]:
    text = safe_string(text)
    found = set()

    for y, m, d in re.findall(r"\b(\d{4})-(\d{2})-(\d{2})\b", text):
        year = int(y)
        month = int(m)
        day = int(d)

        if is_valid_date_parts(day, month, year):
            found.add(f"{year:04d}-{month:02d}-{day:02d}")

    for d, m, y in re.findall(r"\b(\d{1,2})[/-](\d{1,2})[/-](\d{4})\b", text):
        day = int(d)
        month = int(m)
        year = int(y)

        if is_valid_date_parts(day, month, year):
            found.add(f"{day:02d}/{month:02d}/{year:04d}")

    for d, m, y in re.findall(r"\b(\d{1,2})[/-](\d{1,2})[/-](\d{2})\b", text):
        day = int(d)
        month = int(m)
        year = parse_two_digit_year(int(y))

        if is_valid_date_parts(day, month, year):
            found.add(f"{day:02d}/{month:02d}/{year:04d}")

    return sorted(found)


In [153]:
# ============================================================
# Front-zone metadata extraction
# ============================================================
def extract_front_matter(text: str, max_chars: int = 12000) -> str:
    return safe_string(text)[:max_chars]


def get_front_lines(text: str, max_chars: int = 12000, max_lines: int = 80) -> List[str]:
    front = extract_front_matter(text, max_chars=max_chars)
    lines = []
    for line in front.splitlines():
        clean = normalize_whitespace(line)
        if clean:
            lines.append(clean)
    return lines[:max_lines]


def is_valid_person_name(name: str) -> bool:
    name = normalize_whitespace(name)

    if len(name) < 5:
        return False
    if len(name.split()) < 2:
        return False
    if len(name.split()) > 4:
        return False
    if any(ch.isdigit() for ch in name):
        return False
    if re.search(r"[•()\[\]{}:;\\/]", name):
        return False
    if name.isupper():
        return False

    parts = name.split()
    if not all(part[:1].isupper() and not part.isupper() for part in parts if part):
        return False

    banned = {
        "data driven business",
        "university of applied sciences",
        "hogwarts school",
        "harry potter"
    }
    if name.lower() in banned:
        return False

    return True


def has_explicit_title_prefix(line: str) -> bool:
    return any(re.match(p, line, flags=re.IGNORECASE) for p in TITLE_PREFIX_PATTERNS)


def has_explicit_author_prefix(line: str) -> bool:
    return any(re.match(p, line, flags=re.IGNORECASE) for p in AUTHOR_PREFIX_PATTERNS)


def clean_title_candidate(line: str) -> str:
    s = normalize_whitespace(line)

    for pattern in TITLE_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_institution_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS)


def looks_like_title_candidate(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False
    if s.lower() in SECTION_LABELS:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if is_valid_person_name(s):
        return False
    if looks_like_institution_line(s):
        return False
    if is_date_like_line(s):
        return False

    if len(s.split()) < 2:
        return False
    if len(s.split()) > 22:
        return False
    if len(s) > 180:
        return False

    if s.endswith("."):
        return False
    if re.match(r"^(this report|this study|this paper|in this report|recently)\b", s, flags=re.IGNORECASE):
        return False

    return True


def score_title_candidate(line: str, idx: int) -> float:
    s = clean_title_candidate(line)
    score = 0.0

    if not looks_like_title_candidate(s):
        return -999.0

    score += max(0, 15 - idx) * 0.4

    wc = len(s.split())
    if 3 <= wc <= 12:
        score += 2.5
    elif 13 <= wc <= 18:
        score += 1.2

    if has_explicit_title_prefix(line):
        score += 4.0

    words = s.split()
    capitalized_ratio = sum(w[:1].isupper() for w in words if w) / max(len(words), 1)
    score += capitalized_ratio

    if s.isupper():
        score -= 2.0

    if looks_like_institution_line(s):
        score -= 4.0

    if is_date_like_line(s):
        score -= 10.0

    return score


def extract_title(text: str) -> str:
    lines = get_front_lines(text, max_chars=12000, max_lines=80)

    candidates = []
    for idx, line in enumerate(lines[:30]):
        cleaned = clean_title_candidate(line)
        score = score_title_candidate(line, idx)
        if score > -100:
            candidates.append((cleaned, score))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[1], reverse=True)
    best_title, best_score = candidates[0]

    if best_score < 2.5:
        return ""

    return best_title


def extract_names_from_line(line: str) -> List[str]:
    s = normalize_whitespace(line)

    for pattern in AUTHOR_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    names = []
    parts = [normalize_whitespace(p) for p in re.split(r",| and ", s) if normalize_whitespace(p)]

    for part in parts:
        if is_valid_person_name(part):
            names.append(part)

    if not names:
        doc = nlp(s)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                candidate = normalize_whitespace(ent.text)
                if is_valid_person_name(candidate):
                    names.append(candidate)

    unique = []
    seen = set()
    for name in names:
        key = name.lower()
        if key not in seen:
            seen.add(key)
            unique.append(name)

    return unique


def looks_like_author_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False
    if s.lower() in SECTION_LABELS:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if looks_like_institution_line(s):
        return False
    if is_date_like_line(s):
        return False
    if len(s) > 120:
        return False

    if has_explicit_author_prefix(s):
        return True

    if is_valid_person_name(s):
        return True

    parts = [normalize_whitespace(p) for p in re.split(r",| and ", s) if normalize_whitespace(p)]
    valid_parts = [p for p in parts if is_valid_person_name(p)]

    if len(valid_parts) >= 1 and len(valid_parts) == len(parts):
        return True

    return False


def score_author_line(line: str, idx: int) -> float:
    s = normalize_whitespace(line)
    score = 0.0

    if not looks_like_author_line(s):
        return -999.0

    score += max(0, 25 - idx) * 0.2

    if has_explicit_author_prefix(s):
        score += 4.0

    if is_valid_person_name(s):
        score += 2.0

    extracted = extract_names_from_line(s)
    if extracted:
        score += min(len(extracted), 3) * 0.8

    return score


def extract_authors(text: str, max_authors: int = 5) -> List[str]:
    lines = get_front_lines(text, max_chars=12000, max_lines=80)

    candidates = []
    for idx, line in enumerate(lines[:35]):
        score = score_author_line(line, idx)
        if score > -100:
            names = extract_names_from_line(line)
            if names:
                candidates.append((names, score))

    if not candidates:
        return []

    candidates.sort(key=lambda x: x[1], reverse=True)

    final_names = []
    seen = set()

    for names, score in candidates:
        if score < 1.5:
            continue
        for name in names:
            key = name.lower()
            if key not in seen:
                seen.add(key)
                final_names.append(name)

    return final_names[:max_authors]

In [154]:
# ============================================================
# Front-zone metadata extraction
# ============================================================
def extract_front_matter(text: str, max_chars: int = 12000) -> str:
    return safe_string(text)[:max_chars]


def get_front_lines(text: str, max_chars: int = 12000, max_lines: int = 80) -> List[str]:
    front = extract_front_matter(text, max_chars=max_chars)
    lines = []
    for line in front.splitlines():
        clean = normalize_whitespace(line)
        if clean:
            lines.append(clean)
    return lines[:max_lines]


def is_valid_person_name(name: str) -> bool:
    name = normalize_whitespace(name)

    if len(name) < 5:
        return False
    if len(name.split()) < 2:
        return False
    if len(name.split()) > 4:
        return False
    if any(ch.isdigit() for ch in name):
        return False
    if re.search(r"[•()\[\]{}:;\\/]", name):
        return False
    if name.isupper():
        return False

    parts = name.split()
    if not all(part[:1].isupper() and not part.isupper() for part in parts if part):
        return False

    banned = {
        "data driven business",
        "university of applied sciences",
        "hogwarts school",
        "harry potter"
    }
    if name.lower() in banned:
        return False

    return True


def has_explicit_title_prefix(line: str) -> bool:
    return any(re.match(p, line, flags=re.IGNORECASE) for p in TITLE_PREFIX_PATTERNS)


def has_explicit_author_prefix(line: str) -> bool:
    return any(re.match(p, line, flags=re.IGNORECASE) for p in AUTHOR_PREFIX_PATTERNS)


def clean_title_candidate(line: str) -> str:
    s = normalize_whitespace(line)

    for pattern in TITLE_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_institution_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS)


def looks_like_title_candidate(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False
    if s.lower() in SECTION_LABELS:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if is_valid_person_name(s):
        return False
    if looks_like_institution_line(s):
        return False
    if is_date_like_line(s):
        return False

    if len(s.split()) < 2:
        return False
    if len(s.split()) > 22:
        return False
    if len(s) > 180:
        return False

    if s.endswith("."):
        return False
    if re.match(r"^(this report|this study|this paper|in this report|recently)\b", s, flags=re.IGNORECASE):
        return False

    return True


def score_title_candidate(line: str, idx: int) -> float:
    s = clean_title_candidate(line)
    score = 0.0

    if not looks_like_title_candidate(s):
        return -999.0

    score += max(0, 15 - idx) * 0.4

    wc = len(s.split())
    if 3 <= wc <= 12:
        score += 2.5
    elif 13 <= wc <= 18:
        score += 1.2

    if has_explicit_title_prefix(line):
        score += 4.0

    words = s.split()
    capitalized_ratio = sum(w[:1].isupper() for w in words if w) / max(len(words), 1)
    score += capitalized_ratio

    if s.isupper():
        score -= 2.0

    if looks_like_institution_line(s):
        score -= 4.0

    if is_date_like_line(s):
        score -= 10.0

    return score


def extract_title(text: str) -> str:
    lines = get_front_lines(text, max_chars=12000, max_lines=80)

    candidates = []
    for idx, line in enumerate(lines[:30]):
        cleaned = clean_title_candidate(line)
        score = score_title_candidate(line, idx)
        if score > -100:
            candidates.append((cleaned, score))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[1], reverse=True)
    best_title, best_score = candidates[0]

    if best_score < 2.5:
        return ""

    return best_title


def extract_names_from_line(line: str) -> List[str]:
    s = normalize_whitespace(line)

    for pattern in AUTHOR_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    names = []
    parts = [normalize_whitespace(p) for p in re.split(r",| and ", s) if normalize_whitespace(p)]

    for part in parts:
        if is_valid_person_name(part):
            names.append(part)

    if not names:
        doc = nlp(s)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                candidate = normalize_whitespace(ent.text)
                if is_valid_person_name(candidate):
                    names.append(candidate)

    unique = []
    seen = set()
    for name in names:
        key = name.lower()
        if key not in seen:
            seen.add(key)
            unique.append(name)

    return unique


def looks_like_author_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False
    if s.lower() in SECTION_LABELS:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if looks_like_institution_line(s):
        return False
    if is_date_like_line(s):
        return False
    if len(s) > 120:
        return False

    if has_explicit_author_prefix(s):
        return True

    if is_valid_person_name(s):
        return True

    parts = [normalize_whitespace(p) for p in re.split(r",| and ", s) if normalize_whitespace(p)]
    valid_parts = [p for p in parts if is_valid_person_name(p)]

    if len(valid_parts) >= 1 and len(valid_parts) == len(parts):
        return True

    return False


def score_author_line(line: str, idx: int) -> float:
    s = normalize_whitespace(line)
    score = 0.0

    if not looks_like_author_line(s):
        return -999.0

    score += max(0, 25 - idx) * 0.2

    if has_explicit_author_prefix(s):
        score += 4.0

    if is_valid_person_name(s):
        score += 2.0

    extracted = extract_names_from_line(s)
    if extracted:
        score += min(len(extracted), 3) * 0.8

    return score


def extract_authors(text: str, max_authors: int = 5) -> List[str]:
    lines = get_front_lines(text, max_chars=12000, max_lines=80)

    candidates = []
    for idx, line in enumerate(lines[:35]):
        score = score_author_line(line, idx)
        if score > -100:
            names = extract_names_from_line(line)
            if names:
                candidates.append((names, score))

    if not candidates:
        return []

    candidates.sort(key=lambda x: x[1], reverse=True)

    final_names = []
    seen = set()

    for names, score in candidates:
        if score < 1.5:
            continue
        for name in names:
            key = name.lower()
            if key not in seen:
                seen.add(key)
                final_names.append(name)

    return final_names[:max_authors]

In [155]:
# ============================================================
# Research group classification
# ============================================================
def build_research_group_text(
    title: str,
    summary: str,
    summary_input: str,
    raw_text: str,
    max_chars: int = 2500
) -> str:
    pieces = []

    if title:
        pieces.append(f"Title: {title}")
    if summary:
        pieces.append(f"Summary: {summary}")
    if summary_input:
        pieces.append(f"Key content: {summary_input[:1200]}")
    elif raw_text:
        pieces.append(f"Key content: {raw_text[:1200]}")

    text = " ".join(pieces)
    return normalize_whitespace(text)[:max_chars]


def classify_research_group(
    title: str,
    summary: str,
    summary_input: str,
    raw_text: str,
    candidate_labels: List[str],
    top_k: int = 3
) -> Dict[str, Any]:
    classification_text = build_research_group_text(
        title=title,
        summary=summary,
        summary_input=summary_input,
        raw_text=raw_text
    )

    if not classification_text.strip():
        return {
            "research_group": "",
            "research_group_confidence": None,
            "research_group_top3": []
        }

    result = zero_shot_classifier(
        classification_text,
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template="This document best fits the research group {}."
    )

    labels = result["labels"]
    scores = result["scores"]

    top_items = []
    for label, score in list(zip(labels, scores))[:top_k]:
        top_items.append({
            "label": label,
            "score": round(float(score), 6)
        })

    best_label = labels[0] if labels else ""
    best_score = float(scores[0]) if scores else None

    if best_score is not None and best_score < RESEARCH_GROUP_CONFIDENCE_THRESHOLD:
        best_label = ""

    return {
        "research_group": best_label,
        "research_group_confidence": round(best_score, 6) if best_score is not None else None,
        "research_group_top3": top_items
    }

In [156]:
# ============================================================
# Evaluation
# ============================================================
def get_key_entities(text: str, max_entities: int = MAX_KEY_ENTITIES) -> List[str]:
    doc = nlp(safe_string(text)[:4000])

    entities = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "DATE", "PRODUCT", "EVENT"}:
            ent_text = normalize_whitespace(ent.text)
            if len(ent_text) > 2:
                entities.append(ent_text)

    unique = []
    seen = set()
    for ent in entities:
        key = ent.lower()
        if key not in seen:
            seen.add(key)
            unique.append(ent)

    return unique[:max_entities]


def evaluate_summary(
    original_text: str,
    summary_text: str,
    source_entities: Optional[List[str]] = None
) -> Dict[str, Any]:
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)
    compression_ratio = summary_len / original_len if original_len > 0 else 0.0

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    preserved_entities = []
    missed_entities = []

    for ent in source_entities:
        ent_clean = normalize_whitespace(ent)
        if ent_clean.lower() in summary_lower:
            preserved_entities.append(ent_clean)
        else:
            missed_entities.append(ent_clean)

    total_entities = len(source_entities)
    entity_preservation_score = len(preserved_entities) / total_entities if total_entities > 0 else None

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }


def sentence_coverage_score(source_sentences: List[str], summary_text: str, threshold: float = 0.2) -> float:
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue
        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0.0


def summary_redundancy_score(summary_text: str) -> float:
    summary_sentences = split_sentences(summary_text)

    if len(summary_sentences) < 2:
        return 0.0

    similarities = []
    for i in range(len(summary_sentences)):
        for j in range(i + 1, len(summary_sentences)):
            similarities.append(jaccard_similarity(summary_sentences[i], summary_sentences[j]))

    return sum(similarities) / len(similarities) if similarities else 0.0



In [157]:
# ============================================================
# Main document pipeline
# ============================================================
def summarize_document_row(row: pd.Series) -> Dict[str, Any]:
    document_id = safe_string(row.get(DOC_ID_COLUMN, ""))
    text = clean_text(safe_string(row.get(TEXT_COLUMN, "")))

    if not text:
        raise ValueError("Input document has no usable text.")

    if SENTENCES_COLUMN in row and isinstance(row[SENTENCES_COLUMN], list) and row[SENTENCES_COLUMN]:
        sentences = [normalize_whitespace(s) for s in row[SENTENCES_COLUMN] if normalize_whitespace(s)]
    else:
        sentences = split_sentences(text)

    selected_sentences = select_balanced_sentences(sentences)
    summary_input = build_summary_input(selected_sentences)

    if not summary_input:
        summary_input = normalize_whitespace(text[:4000])

    key_entities = get_key_entities(summary_input)

    start = time.time()
    summary = summarize_mbart(summary_input)
    runtime_seconds = time.time() - start

    summary = clean_generated_summary(summary)

    eval_result = evaluate_summary(
        original_text=text,
        summary_text=summary,
        source_entities=key_entities
    )
    eval_result["runtime_seconds"] = runtime_seconds
    eval_result["sentence_coverage_score"] = sentence_coverage_score(selected_sentences, summary)
    eval_result["summary_redundancy_score"] = summary_redundancy_score(summary)

    metadata = extract_generic_metadata(
        text=text,
        summary=summary,
        summary_input=summary_input,
        document_id=document_id
    )

    return {
        "document_id": document_id,
        "model": "mBART_single_input_general_reproducible_v6",
        "selected_sentences": selected_sentences,
        "summary_input": summary_input,
        "summary": summary,
        "metadata": metadata,
        "evaluation": {
            "runtime_seconds": eval_result["runtime_seconds"],
            "original_token_count": eval_result["original_token_count"],
            "summary_token_count": eval_result["summary_token_count"],
            "compression_ratio": eval_result["compression_ratio"],
            "entity_preservation_score": eval_result["entity_preservation_score"],
            "sentence_coverage_score": eval_result["sentence_coverage_score"],
            "summary_redundancy_score": eval_result["summary_redundancy_score"],
            "preserved_entities": eval_result["preserved_entities"],
            "missed_entities": eval_result["missed_entities"]
        }
    }


In [158]:
# ============================================================
# mBART summarization
# ============================================================
def summarize_mbart(
    text: str,
    max_input_len: int = MAX_INPUT_LEN,
    max_output_len: int = MAX_OUTPUT_LEN,
    min_output_len: int = MIN_OUTPUT_LEN
) -> str:
    inputs = tokenizer(
        safe_string(text),
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=6,
            length_penalty=1.25,
            no_repeat_ngram_size=4,
            repetition_penalty=1.25,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def clean_generated_summary(summary: str) -> str:
    summary = remove_section_prefix(summary)
    summary = re.sub(r"\s+", " ", summary).strip()
    return summary



In [161]:
# ============================================================
# Run one document
# ============================================================
def main() -> None:
    df = pd.read_json(INPUT_JSON, orient="records")

    if df.empty:
        raise ValueError("Input JSON is empty.")

    row = df.loc[0]
    result = summarize_document_row(row)

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / OUTPUT_FILE
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4, ensure_ascii=False)

    print(f"Saved combined mBART output to {output_path}")
    print(result["summary"])
    print(json.dumps(result["metadata"], indent=4, ensure_ascii=False))


if __name__ == "__main__":
    main()

Saved combined mBART output to ..\Data\gold\mBART\mBART_output.json
This report examines how the magical institution of Hogwarts can become data driven. First, this study explores core principles of data driven decision making, where we cover data literacy, culture, and governance. Finally, we lay out a strategy for hogwarts, outlining objectives, steps, challenges timeline, resource allocation, and success metrics. The report demonstrates that becoming data driven requires technological solutions, strong governance and ethical responsibility. Recently organizations have relied more on data to support their decision making. In this report we focus on the concept of being a data driven business. We apply this to a fictional case, based on Harry Potter.
{
    "id": "doc_02",
    "title": "1. Introduction",
    "title_found": true,
    "description": "This report examines how the magical institution of Hogwarts can become data driven. First, this study explores core principles of data dri

In [163]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# ============================================================
# Configuration
# ============================================================
INPUT_JSON = "../Data/silver/doc_01_silver_nlp.json"
OUTPUT_DIR = "../Data/gold/mBART"
OUTPUT_FILE = "mBART_output.json"

TEXT_COLUMN = "cleaned_text"
RAW_TEXT_COLUMN = "raw_text"
DOC_ID_COLUMN = "document_id"
SENTENCES_COLUMN = "sentences"

MBART_MODEL = "facebook/mbart-large-50"
SRC_LANG = "en_XX"

ZERO_SHOT_MODEL = "facebook/bart-large-mnli"

MAX_INPUT_LEN = 1024
MAX_OUTPUT_LEN = 170
MIN_OUTPUT_LEN = 110

MAX_SIMILARITY = 0.68
MAX_SELECTED_SENTENCES = 12
MAX_KEY_ENTITIES = 15

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoiveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology",
]
RESEARCH_GROUP_CONFIDENCE_THRESHOLD = 0.20

# ============================================================
# Load models
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    MBART_MODEL,
    src_lang=SRC_LANG,
    use_fast=False
)
model = AutoModelForSeq2SeqLM.from_pretrained(MBART_MODEL).to(device)
model.eval()

nlp = spacy.load("en_core_web_sm")

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model=ZERO_SHOT_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

# ============================================================
# Generic helpers
# ============================================================
SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents",
    "table of contents"
}

INSTITUTION_PATTERNS = [
    r"\buniversity\b",
    r"\bcollege\b",
    r"\bfaculty\b",
    r"\bschool\b",
    r"\binstitute\b",
    r"\bdepartment\b",
    r"\bapplied sciences\b",
    r"\bhz university\b"
]

MONTH_WORDS = (
    "january|february|march|april|may|june|july|august|september|october|november|december|"
    "jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec"
)

DATE_PATTERN = (
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|"
    r"\b\d{4}-\d{2}-\d{2}\b|"
    rf"\b\d{{1,2}}\s+(?:{MONTH_WORDS})\s+\d{{2,4}}\b|"
    rf"\b(?:{MONTH_WORDS})\s+\d{{1,2}},?\s+\d{{2,4}}\b"
)

COURSE_CODE_PATTERN = r"\b[A-Z]{2,6}\d{4,}[A-Z0-9]*\b"

AUTHOR_PREFIX_PATTERNS = [
    r"^(author|authors|written by|student|students|name|contributors?)\s*[:\-]?\s+",
    r"^by\s+[A-Z]"
]


def safe_string(value: Any) -> str:
    return "" if value is None else str(value)


def normalize_whitespace(text: Any) -> str:
    return re.sub(r"\s+", " ", safe_string(text)).strip()


def simple_tokenize(text: Any) -> List[str]:
    return re.findall(r"\b\w+\b", safe_string(text).lower())


def dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        clean = normalize_whitespace(item)
        key = clean.lower()
        if clean and key not in seen:
            seen.add(key)
            out.append(clean)
    return out


def clean_text(text: str) -> str:
    text = safe_string(text)
    text = text.replace("\x00", " ").replace("\ufeff", " ")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    cleaned_lines = []
    for line in text.splitlines():
        line_strip = normalize_whitespace(line)

        if re.fullmatch(r"\d{1,4}", line_strip):
            continue
        if re.fullmatch(r"page\s+\d+(\s+of\s+\d+)?", line_strip, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"\d+\s*\|\s*p\s*a\s*g\s*e", line_strip, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"\d+\s*\|\s*page", line_strip, flags=re.IGNORECASE):
            continue

        cleaned_lines.append(line.strip())

    return "\n".join([x for x in cleaned_lines if x]).strip()


def split_sentences(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    doc = nlp(text)
    sentences = [normalize_whitespace(sent.text) for sent in doc.sents]
    return [s for s in sentences if s]

# ============================================================
# Noise / heading detection
# ============================================================
def remove_section_prefix(sentence: str) -> str:
    s = normalize_whitespace(sentence)
    s = re.sub(
        r"^(abstract|introduction|background|discussion|conclusion|conclusions|results|result|method|methods|methodology|summary)\b[:\-\s]*",
        "",
        s,
        flags=re.IGNORECASE
    )
    return normalize_whitespace(s)


def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if "table of contents" in s.lower():
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.search(r"\.{4,}", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    if re.search(r"\b\d+\.\d+\b", s) and len(s) < 120:
        return True

    return False


def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True

    return False


def is_heading_like(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if len(s.split()) <= 5 and s.istitle():
        return True
    if re.fullmatch(r"[A-Z][A-Za-z\s/&\-]{1,60}", s) and len(s.split()) <= 8:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True

    return False


def clean_sentence_for_selection(sentence: str) -> str:
    s = remove_section_prefix(sentence)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_good_sentence(sentence: str) -> bool:
    s = clean_sentence_for_selection(sentence)

    if len(s) < 45:
        return False
    if len(s.split()) < 8:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if is_heading_like(s):
        return False
    if re.match(r"^\d+(\.\d+)*\s*$", s):
        return False
    if "................................................................" in s:
        return False
    if "•" in s and len(s.split()) < 15:
        return False

    return True

# ============================================================
# Ranking and sentence selection
# ============================================================
def jaccard_similarity(a: str, b: str) -> float:
    set_a = set(simple_tokenize(a))
    set_b = set(simple_tokenize(b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def sentence_position_score(index: int, total: int) -> float:
    if total <= 0:
        return 0.0

    relative = index / max(total, 1)

    if relative < 0.10:
        return 0.9
    if relative < 0.25:
        return 0.5
    if relative > 0.90:
        return 0.4

    return 0.0


def sentence_information_score(sentence: str) -> float:
    s = sentence.lower()
    word_count = len(sentence.split())
    score = 0.0

    if 12 <= word_count <= 35:
        score += 1.0
    elif 36 <= word_count <= 50:
        score += 0.6
    elif word_count > 55:
        score -= 0.5

    discourse_terms = [
        "this study", "this report", "this paper", "the aim", "the purpose",
        "the objective", "the analysis", "the findings", "the results",
        "shows that", "demonstrates that", "indicates that", "suggests that",
        "because", "therefore", "however", "overall", "in conclusion"
    ]
    for term in discourse_terms:
        if term in s:
            score += 0.5

    tokens = simple_tokenize(sentence)
    if tokens:
        unique_ratio = len(set(tokens)) / len(tokens)
        score += 0.8 * unique_ratio

    vague_phrases = [
        "this is needed", "it also makes", "it means that", "there are a lot of"
    ]
    for phrase in vague_phrases:
        if phrase in s:
            score -= 0.5

    return score


def classify_sentence_role(sentence: str) -> str:
    s = sentence.lower()

    if any(x in s for x in [
        "this study", "this report", "this paper", "the aim", "the purpose",
        "we examine", "we explore", "we investigate", "focus on", "focuses on"
    ]):
        return "objective"

    if any(x in s for x in [
        "method", "approach", "analysis", "using", "based on",
        "evaluate", "evaluates", "assessment", "assesses", "analyzes", "examines"
    ]):
        return "method"

    if any(x in s for x in [
        "results", "findings", "shows that", "demonstrates that",
        "indicates that", "reveals that", "suggests that", "is reduced"
    ]):
        return "findings"

    if any(x in s for x in [
        "recommend", "should", "could", "future work", "implementation",
        "strategy", "plan", "timeline", "metrics for success"
    ]):
        return "recommendation"

    if any(x in s for x in [
        "in conclusion", "overall", "to conclude", "in summary"
    ]):
        return "conclusion"

    return "background"


def rank_sentences(sentences: List[str]) -> List[Tuple[str, float]]:
    cleaned_candidates = []
    for i, s in enumerate(sentences):
        cleaned = clean_sentence_for_selection(s)
        if is_good_sentence(cleaned):
            cleaned_candidates.append((i, cleaned))

    total = len(cleaned_candidates)
    scored = []

    for i, sentence in cleaned_candidates:
        score = sentence_information_score(sentence) + sentence_position_score(i, total)
        scored.append((sentence, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


def select_balanced_sentences(
    sentences: List[str],
    max_sentences: int = MAX_SELECTED_SENTENCES,
    max_similarity: float = MAX_SIMILARITY,
    max_input_tokens: int = MAX_INPUT_LEN
) -> List[str]:
    ranked = rank_sentences(sentences)

    buckets = {
        "objective": [],
        "method": [],
        "findings": [],
        "recommendation": [],
        "conclusion": [],
        "background": []
    }

    for sent, _ in ranked:
        buckets[classify_sentence_role(sent)].append(sent)

    selected = []
    target_plan = [
        ("objective", 2),
        ("method", 2),
        ("findings", 2),
        ("recommendation", 2),
        ("conclusion", 1),
        ("background", 3),
    ]

    for role, limit in target_plan:
        count = 0
        for sent in buckets[role]:
            if any(jaccard_similarity(sent, prev) > max_similarity for prev in selected):
                continue

            candidate_text = " ".join(selected + [sent])
            token_count = len(tokenizer(candidate_text, truncation=False)["input_ids"])
            if token_count > max_input_tokens - 30:
                continue

            selected.append(sent)
            count += 1
            if count >= limit or len(selected) >= max_sentences:
                break

    if len(selected) < max_sentences:
        for sent, _ in ranked:
            if sent in selected:
                continue
            if any(jaccard_similarity(sent, prev) > max_similarity for prev in selected):
                continue

            candidate_text = " ".join(selected + [sent])
            token_count = len(tokenizer(candidate_text, truncation=False)["input_ids"])
            if token_count > max_input_tokens - 30:
                continue

            selected.append(sent)
            if len(selected) >= max_sentences:
                break

    ordered = []
    cleaned_source = [clean_sentence_for_selection(s) for s in sentences]
    selected_set = set(selected)

    for s in cleaned_source:
        if s in selected_set and s not in ordered:
            ordered.append(s)

    return ordered[:max_sentences]


def build_summary_input(selected_sentences: List[str]) -> str:
    cleaned = []

    for s in selected_sentences:
        s = clean_sentence_for_selection(s)
        if is_heading_like(s):
            continue
        cleaned.append(s)

    text = " ".join(cleaned)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ============================================================
# Date helpers
# ============================================================
def parse_two_digit_year(year_2d: int) -> int:
    if 0 <= year_2d <= 30:
        return 2000 + year_2d
    return 1900 + year_2d


def is_valid_date_parts(day: int, month: int, year: int) -> bool:
    current_year = time.localtime().tm_year

    if month < 1 or month > 12:
        return False
    if day < 1 or day > 31:
        return False
    if year < 1900 or year > current_year + 5:
        return False

    return True


def is_date_like_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False

    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\s*[-–]\s*\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}\s*[-–]\s*\d{4}-\d{2}-\d{2}", s):
        return True

    return False


def extract_dates(text: str) -> List[str]:
    text = safe_string(text)
    found = []

    for match in re.findall(DATE_PATTERN, text, flags=re.IGNORECASE):
        clean = normalize_whitespace(match)
        if clean:
            found.append(clean)

    return dedupe_keep_order(found)


def parse_date_to_sortable(date_str: str) -> Tuple[int, int, int]:
    s = normalize_whitespace(date_str)

    m = re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", s)
    if m:
        day = int(m.group(1))
        month = int(m.group(2))
        year = int(m.group(3))
        if year < 100:
            year = parse_two_digit_year(year)
        return (year, month, day)

    m = re.fullmatch(r"(\d{4})-(\d{2})-(\d{2})", s)
    if m:
        return (int(m.group(1)), int(m.group(2)), int(m.group(3)))

    return (9999, 99, 99)

# ============================================================
# Front matter and metadata extraction
# ============================================================
def extract_front_matter(text: str, max_chars: int = 15000) -> str:
    text = safe_string(text)
    if not text:
        return ""

    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)

    toc_match = re.search(r"\btable\s+of\s+contents\b", text, flags=re.IGNORECASE)
    if toc_match:
        return text[:toc_match.start()].strip()

    abstract_match = re.search(r"\babstract\b", text, flags=re.IGNORECASE)
    if abstract_match and abstract_match.start() > 0:
        # keep cover page only
        return text[:abstract_match.start()].strip()

    return text[:max_chars].strip()


def get_front_lines_raw(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    front = extract_front_matter(text, max_chars=max_chars)
    lines = []
    for line in front.splitlines():
        raw = safe_string(line).strip()
        if raw:
            lines.append(raw)
    return lines[:max_lines]


def split_joined_front_line(line: str) -> List[str]:
    """
    Recover structure from flattened cover-page lines like:
    'HZ University of Applied Sciences Data Driven Business (CU75072V2) DATA DRIVEN BUSINESS Lars de Bruijne, Joeri Harreman'
    """
    s = normalize_whitespace(line)
    if not s:
        return []

    parts: List[str] = []

    # First, try to extract trailing author block
    author_match = re.search(
        r"([A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+"
        r"(?:\s*,\s*[A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+)*)$",
        s
    )

    author_part = ""
    if author_match:
        author_part = author_match.group(1).strip()
        s = s[:author_match.start()].strip()

    # Split around course code parentheses, e.g. (CU75072V2)
    code_match = re.search(r"\([A-Z]{2,6}\d{4,}[A-Z0-9]*\)", s)
    if code_match:
        left = s[:code_match.end()].strip()
        right = s[code_match.end():].strip()
        if left:
            parts.append(left)
        if right:
            parts.append(right)
    else:
        parts.append(s)

    # Split full-uppercase segment if present
    final_parts: List[str] = []
    for part in parts:
        part = normalize_whitespace(part)
        if not part:
            continue

        # capture ALL CAPS title-like chunk, e.g. DATA DRIVEN BUSINESS
        caps_match = re.search(r"\b([A-Z][A-Z\s&\-]{5,})\b", part)
        if caps_match and caps_match.group(1).strip() != part.strip():
            before = normalize_whitespace(part[:caps_match.start()])
            caps = normalize_whitespace(caps_match.group(1))
            after = normalize_whitespace(part[caps_match.end():])

            if before:
                final_parts.append(before)
            if caps:
                final_parts.append(caps)
            if after:
                final_parts.append(after)
        else:
            final_parts.append(part)

    if author_part:
        final_parts.append(author_part)

    return [p for p in final_parts if p]


def get_front_lines(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    raw_lines = get_front_lines_raw(text, max_chars=max_chars, max_lines=max_lines)

    processed: List[str] = []
    for line in raw_lines:
        split_parts = split_joined_front_line(line)
        if split_parts:
            processed.extend(split_parts)
        else:
            processed.append(normalize_whitespace(line))

    return [normalize_whitespace(x) for x in processed if normalize_whitespace(x)][:max_lines]


def is_valid_person_name(name: str) -> bool:
    name = normalize_whitespace(name)

    if len(name) < 5:
        return False
    if len(name.split()) < 2:
        return False
    if len(name.split()) > 5:
        return False
    if any(ch.isdigit() for ch in name):
        return False
    if re.search(r"[•()\[\]{}:;\\/]", name):
        return False

    banned = {
        "data driven business",
        "hz university of applied sciences",
        "table of contents",
        "abstract",
        "harry potter",
        "hogwarts"
    }
    if name.lower() in banned:
        return False

    parts = name.split()
    lower_particles = {"de", "den", "der", "van", "von", "ten", "ter", "op", "la", "le", "du", "di"}

    proper_like = 0
    for i, part in enumerate(parts):
        if i > 0 and part.lower() in lower_particles:
            proper_like += 1
            continue
        if part[:1].isupper() and not part.isupper():
            proper_like += 1

    return proper_like >= 2


def extract_contributors_from_line(line: str) -> List[str]:
    s = normalize_whitespace(line)
    if not s:
        return []

    for pattern in AUTHOR_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    parts = [normalize_whitespace(p) for p in re.split(r",| and | & |;", s) if normalize_whitespace(p)]
    names = [p for p in parts if is_valid_person_name(p)]
    if names:
        return dedupe_keep_order(names)

    doc = nlp(s)
    ents = []
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            candidate = normalize_whitespace(ent.text)
            if is_valid_person_name(candidate):
                ents.append(candidate)

    return dedupe_keep_order(ents)


def extract_contributors(text: str, max_authors: int = 8) -> List[str]:
    lines = get_front_lines(text, max_chars=15000, max_lines=40)

    found = []
    for line in lines:
        names = extract_contributors_from_line(line)
        if names:
            found.extend(names)

    return dedupe_keep_order(found)[:max_authors]


def is_title_noise(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if is_table_of_contents_line(s):
        return True
    if looks_like_reference(s):
        return True
    if is_date_like_line(s):
        return True
    if re.fullmatch(r"\d+\s*\|\s*p\s*a\s*g\s*e", s, flags=re.IGNORECASE):
        return True
    if re.fullmatch(
        r"\d+(\.\d+)*\s*(introduction|background|methods?|results?|discussion|conclusions?|references|appendix)",
        s,
        flags=re.IGNORECASE
    ):
        return True
    return False


def is_likely_org_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS)


def is_likely_course_or_module_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if re.search(r"\([A-Z]{2,6}\d{4,}[A-Z0-9]*\)", s):
        return True
    if "data driven business" in s.lower():
        return True
    if any(k in s.lower() for k in ["course", "module", "minor", "program", "track"]):
        return True
    return False


def score_title_candidate(line: str, contributors: List[str]) -> float:
    s = normalize_whitespace(line)

    if is_title_noise(s):
        return -999.0
    if any(s.lower() == c.lower() for c in contributors):
        return -999.0

    score = 0.0

    wc = len(s.split())
    if wc < 2 or wc > 16:
        return -999.0

    if is_likely_org_line(s):
        score -= 3.0
    if is_likely_course_or_module_line(s):
        score += 2.5

    # Favour short title-like cover page lines
    if 2 <= wc <= 8:
        score += 2.0
    elif 9 <= wc <= 14:
        score += 1.0

    # Favour uppercase standalone cover titles
    alpha = [ch for ch in s if ch.isalpha()]
    if alpha:
        uppercase_ratio = sum(ch.isupper() for ch in alpha) / len(alpha)
        if uppercase_ratio > 0.85:
            score += 1.2

    # Penalize commas because that often signals author list
    if "," in s:
        score -= 2.0

    # Penalize lines that are mostly institution + title blended together
    if is_likely_org_line(s) and is_likely_course_or_module_line(s):
        score -= 1.5

    return score


def extract_title(text: str, contributors: Optional[List[str]] = None) -> str:
    contributors = contributors or []
    lines = get_front_lines(text, max_chars=15000, max_lines=30)

    candidates = []
    for line in lines:
        score = score_title_candidate(line, contributors)
        if score > -100:
            candidates.append((line, score))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[1], reverse=True)
    best_title, best_score = candidates[0]

    if best_score < 1.5:
        return ""

    return best_title


def extract_dates_in_order(text: str) -> List[str]:
    return extract_dates(text)


def extract_start_end_dates(text: str) -> Tuple[str, str, List[str]]:
    found = extract_dates_in_order(extract_front_matter(text))

    if not found:
        return "", "", []

    if len(found) == 1:
        return found[0], "", found

    return found[0], found[-1], found


def extract_abstract_text(text: str, max_chars: int = 1600) -> str:
    text = safe_string(text)

    match = re.search(
        r"\babstract\b\s*(.*?)(?=\n\s*(table\s+of\s+contents|\d+(\.\d+)*\s+[A-Za-z]|introduction)\b|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if not match:
        return ""

    abstract = match.group(1)
    abstract = clean_text(abstract)
    abstract = re.sub(r"\s+", " ", abstract).strip()
    return abstract[:max_chars]

# ============================================================
# Research group classification
# ============================================================
def build_research_group_text(
    title: str,
    summary: str,
    summary_input: str,
    raw_text: str,
    max_chars: int = 2500
) -> str:
    pieces = []

    if title:
        pieces.append(f"Title: {title}")
    if summary:
        pieces.append(f"Summary: {summary}")
    if summary_input:
        pieces.append(f"Key content: {summary_input[:1200]}")
    elif raw_text:
        pieces.append(f"Key content: {raw_text[:1200]}")

    text = " ".join(pieces)
    return normalize_whitespace(text)[:max_chars]


def classify_research_group(
    title: str,
    summary: str,
    summary_input: str,
    raw_text: str,
    candidate_labels: List[str],
    top_k: int = 3
) -> Dict[str, Any]:
    classification_text = build_research_group_text(
        title=title,
        summary=summary,
        summary_input=summary_input,
        raw_text=raw_text
    )

    if not classification_text.strip():
        return {
            "research_group": "",
            "research_group_confidence": None,
            "research_group_top3": []
        }

    result = zero_shot_classifier(
        classification_text,
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template="This document best fits the research group {}."
    )

    labels = result["labels"]
    scores = result["scores"]

    top_items = []
    for label, score in list(zip(labels, scores))[:top_k]:
        top_items.append({
            "label": label,
            "score": round(float(score), 6)
        })

    best_label = labels[0] if labels else ""
    best_score = float(scores[0]) if scores else None

    if best_score is not None and best_score < RESEARCH_GROUP_CONFIDENCE_THRESHOLD:
        best_label = ""

    return {
        "research_group": best_label,
        "research_group_confidence": round(best_score, 6) if best_score is not None else None,
        "research_group_top3": top_items
    }

# ============================================================
# Evaluation
# ============================================================
def get_key_entities(text: str, max_entities: int = MAX_KEY_ENTITIES) -> List[str]:
    doc = nlp(safe_string(text)[:12000])

    entities = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "DATE", "PRODUCT", "EVENT"}:
            ent_text = normalize_whitespace(ent.text)
            if len(ent_text) > 2:
                entities.append(ent_text)

    return dedupe_keep_order(entities)[:max_entities]


def evaluate_summary(
    original_text: str,
    summary_text: str,
    source_entities: Optional[List[str]] = None
) -> Dict[str, Any]:
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)
    compression_ratio = summary_len / original_len if original_len > 0 else 0.0

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    preserved_entities = []
    missed_entities = []

    for ent in source_entities:
        ent_clean = normalize_whitespace(ent)
        if ent_clean.lower() in summary_lower:
            preserved_entities.append(ent_clean)
        else:
            missed_entities.append(ent_clean)

    total_entities = len(source_entities)
    entity_preservation_score = len(preserved_entities) / total_entities if total_entities > 0 else None

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }


def sentence_coverage_score(source_sentences: List[str], summary_text: str, threshold: float = 0.2) -> float:
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue
        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0.0


def summary_redundancy_score(summary_text: str) -> float:
    summary_sentences = split_sentences(summary_text)

    if len(summary_sentences) < 2:
        return 0.0

    similarities = []
    for i in range(len(summary_sentences)):
        for j in range(i + 1, len(summary_sentences)):
            similarities.append(jaccard_similarity(summary_sentences[i], summary_sentences[j]))

    return sum(similarities) / len(similarities) if similarities else 0.0

# ============================================================
# Metadata extraction
# ============================================================
def extract_generic_metadata(
    text: str,
    summary: str,
    summary_input: str,
    document_id: str
) -> Dict[str, Any]:
    cleaned_text = clean_text(text)

    contributors = extract_contributors(cleaned_text)
    title = extract_title(cleaned_text, contributors=contributors)
    start_date, end_date, all_dates = extract_start_end_dates(cleaned_text)

    abstract_text = extract_abstract_text(cleaned_text)
    description = abstract_text or summary or summary_input[:500]

    research_group_info = classify_research_group(
        title=title,
        summary=summary,
        summary_input=summary_input,
        raw_text=cleaned_text,
        candidate_labels=RESEARCH_GROUPS
    )

    return {
        "id": document_id,
        "title": title,
        "title_found": bool(title),
        "contributors": contributors,
        "contributors_found": bool(contributors),
        "start_date": start_date,
        "end_date": end_date,
        "dates_found": all_dates,
        "description": description,
        "research_group": research_group_info["research_group"],
        "research_group_confidence": research_group_info["research_group_confidence"],
        "research_group_top3": research_group_info["research_group_top3"],
        "metadata_debug": {
            "front_lines_sample": get_front_lines(cleaned_text)[:12]
        }
    }

# ============================================================
# mBART summarization
# ============================================================
def summarize_mbart(
    text: str,
    max_input_len: int = MAX_INPUT_LEN,
    max_output_len: int = MAX_OUTPUT_LEN,
    min_output_len: int = MIN_OUTPUT_LEN
) -> Tuple[str, int, int]:
    prompt_text = f"Summarize the following document: {safe_string(text)}"

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )
    input_token_count = int(inputs["input_ids"].shape[1])
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=4,
            length_penalty=1.45,
            no_repeat_ngram_size=5,
            repetition_penalty=1.25,
            early_stopping=True
        )

    output_token_count = int(summary_ids.shape[1])
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary, input_token_count, output_token_count


def clean_generated_summary(summary: str) -> str:
    summary = remove_section_prefix(summary)
    summary = re.sub(r"\s+", " ", summary).strip()
    return summary

# ============================================================
# Main document pipeline
# ============================================================
def summarize_document_row(row: pd.Series) -> Dict[str, Any]:
    document_id = safe_string(row.get(DOC_ID_COLUMN, ""))

    text_source = row.get(TEXT_COLUMN)
    if not safe_string(text_source).strip():
        text_source = row.get(RAW_TEXT_COLUMN, "")

    text = clean_text(safe_string(text_source))

    if not text:
        raise ValueError("Input document has no usable text.")

    if SENTENCES_COLUMN in row and isinstance(row[SENTENCES_COLUMN], list) and row[SENTENCES_COLUMN]:
        sentences = [normalize_whitespace(s) for s in row[SENTENCES_COLUMN] if normalize_whitespace(s)]
    else:
        sentences = split_sentences(text)

    selected_sentences = select_balanced_sentences(sentences)
    summary_input = build_summary_input(selected_sentences)

    if not summary_input:
        summary_input = normalize_whitespace(text[:4000])

    key_entities = get_key_entities(text[:12000])

    start = time.time()
    summary, input_token_count, output_token_count = summarize_mbart(summary_input)
    runtime_seconds = time.time() - start

    summary = clean_generated_summary(summary)

    eval_result = evaluate_summary(
        original_text=text,
        summary_text=summary,
        source_entities=key_entities
    )
    eval_result["runtime_seconds"] = runtime_seconds
    eval_result["sentence_coverage_score"] = sentence_coverage_score(selected_sentences, summary)
    eval_result["summary_redundancy_score"] = summary_redundancy_score(summary)
    eval_result["input_token_count_to_model"] = input_token_count
    eval_result["output_token_count_from_model"] = output_token_count

    metadata = extract_generic_metadata(
        text=text,
        summary=summary,
        summary_input=summary_input,
        document_id=document_id
    )

    return {
        "document_id": document_id,
        "model": "mBART_single_input_general_reproducible_v8",
        "selected_sentences": selected_sentences,
        "summary_input": summary_input,
        "summary": summary,
        "metadata": metadata,
        "evaluation": {
            "runtime_seconds": eval_result["runtime_seconds"],
            "original_token_count": eval_result["original_token_count"],
            "summary_token_count": eval_result["summary_token_count"],
            "compression_ratio": eval_result["compression_ratio"],
            "entity_preservation_score": eval_result["entity_preservation_score"],
            "sentence_coverage_score": eval_result["sentence_coverage_score"],
            "summary_redundancy_score": eval_result["summary_redundancy_score"],
            "input_token_count_to_model": eval_result["input_token_count_to_model"],
            "output_token_count_from_model": eval_result["output_token_count_from_model"],
            "preserved_entities": eval_result["preserved_entities"],
            "missed_entities": eval_result["missed_entities"]
        }
    }

# ============================================================
# Run one document
# ============================================================
def main() -> None:
    df = pd.read_json(INPUT_JSON, orient="records")

    if df.empty:
        raise ValueError("Input JSON is empty.")

    row = df.loc[0]
    result = summarize_document_row(row)

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / OUTPUT_FILE
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4, ensure_ascii=False)

    print(f"Saved combined mBART output to {output_path}")
    print("\nSUMMARY:\n")
    print(result["summary"])
    print("\nMETADATA:\n")
    print(json.dumps(result["metadata"], indent=4, ensure_ascii=False))


if __name__ == "__main__":
    main()

Loading weights: 100%|██████████| 519/519 [00:00<00:00, 174678.52it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 10232.34it/s]


Saved combined mBART output to ..\Data\gold\mBART\mBART_output.json

SUMMARY:

Summarize the following document: This report examines how the magical institution of Hogwarts can become data driven. First, this study explores core principles of data driven decision making, where we cover data literacy, culture, and governance. Finally, we lay out a strategy for Hogwarts, outlining objectives, steps, challenges timeline, resource allocation, and success metrics. The report demonstrates that becoming data driven requires technological solutions, strong governance and ethical responsibility. Recently organizations have relied more on data to support their decision making. In this report we focus on the concept of being a data driven business.

METADATA:

{
    "id": "doc_02",
    "title": "DATA DRIVEN BUSINESS",
    "title_found": true,
    "contributors": [
        "Lars de Bruijne",
        "Joeri Harreman"
    ],
    "contributors_found": true,
    "start_date": "01/04/2005",
    "end_d